# Laboratorio 1 - AlpesHearth

## Integrantes

- Isabella Naranjo
- David Caro

## Exploración de los datos

Durante la fase de exploración de datos encontramos las siguientes irregularidades en el set de datos:
1. Existen 14 categorias que tienen registros faltantes, algunas de estas son la edad, el peso, la altura y el índice de masa corporal (BMI).
2. Las categorias booleanas vienen con strings como valor, es decir, "Y/N", en vez de True/False.
3. Existen registros de la fecha de servicio (Date of Service) que tienen formatos inconsistentes, es decir, vienen en múltiples formatos.
4. Existen algunos valores imposibles en las categorias de edad, peso, BMI, CVD Risk Score y Estimated LDL.
5. Existen 2 columnas de altura, una en metros y la otra en centimetros, esta información es la misma y puede ser redundante.

Ahora bien, para realizar una exploración más profunda utilizamos la librería pandas para entender la estructura general de los datos.

In [2]:
import pandas as pd 

df = pd.read_csv("../data/Datos Lab 1.csv")
print("Primeras 5 filas del DataFrame: ")
print(df.head())
print("\nUltimas 5 filas del DataFrame: ")
print(df.tail())
print("\nDimensiones del DataFrame")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Total de celdas: {df.size}")

Primeras 5 filas del DataFrame: 
  Patient ID    Date of Service Sex   Age  Weight (kg)  Height (m)     BMI  \
0   isDx5313  November 08, 2023   M  44.0      114.300       1.720  38.600   
1   LHCK2961         20/03/2024   F  57.0       92.923       1.842  33.116   
2   WjVn1699         2021-05-27   F   NaN       73.400       1.650  27.000   
3   dCDO1109     April 18, 2022   F  35.0      113.300       1.780  35.800   
4   pnpE1080         01/11/2024   F  48.0      102.200       1.750  33.400   

   Abdominal Circumference (cm) Blood Pressure (mmHg)  \
0                       100.000                112/83   
1                       106.315                101/91   
2                        78.100                 90/74   
3                        79.600                 92/89   
4                       106.700                121/68   

   Total Cholesterol (mg/dL)  ...  Physical Activity Level  \
0                      228.0  ...                     High   
1                      158.0  .

In [14]:
df.info()
columnas_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
columnas_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nColumnas Numericas ({len(columnas_numericas)}): ")
print(columnas_numericas)
print(f"\nColumnas Categoricas ({len(columnas_categoricas)}): ")
print(columnas_categoricas)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639 entries, 0 to 1638
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Patient ID                    1639 non-null   object 
 1   Date of Service               1639 non-null   object 
 2   Sex                           1639 non-null   object 
 3   Age                           1571 non-null   float64
 4   Weight (kg)                   1566 non-null   float64
 5   Height (m)                    1578 non-null   float64
 6   BMI                           1586 non-null   float64
 7   Abdominal Circumference (cm)  1578 non-null   float64
 8   Blood Pressure (mmHg)         1639 non-null   object 
 9   Total Cholesterol (mg/dL)     1571 non-null   float64
 10  HDL (mg/dL)                   1557 non-null   float64
 11  Fasting Blood Sugar (mg/dL)   1585 non-null   float64
 12  Smoking Status                1639 non-null   object 
 13  Dia

De estas líneas de código pudimos identificar lo siguiente respecto a la estructura de los datos:

1. Tenemos 1639 registros y 24 columnas (14 númericas y 10 categoricas).
2. Efectivamente hay valores nulos en las columnas de edad, peso, altura (m), BMI, circumferencia abdominal, colesterol total, HDL, glucosa en ayunas, altura (cm), relación cintura-altura, presión arterial sistolica, presión arterial diastólica, LDL estimado y puntaje de riesgo cardiovascular (CVD Risk Score).
3. Hay 29 valores nulos en nuestra variable objetivo **CVD Risk Score**.
4. La variable **Date of Service** aparece como tipo object dado que viene en diferentes formatos, sin embargo, fue documentada como Date y es necesario hacer un casting para poder manipularla.
5. Las variables **Systolic BP** y **Diastolic BP** son las componentes de la variable **Blood Pressure**, así que puede que podamos descarta dicha columna. 
6. La variable **CVD Risk Level** es categórica y probablemente sea derivada del **CVD Risk Score**, por lo que no será utilizada como variable predictora para evitar fuga de información.

Seguido a esto, corrimos las siguientes líneas de código, para identificar las irregularidades especificas de las diferentes categorias.

In [13]:
registros_duplicados = df.duplicated().sum()
print(f"Registros duplicados del data set: {registros_duplicados} ({(registros_duplicados * 100 / df.shape[0]).round(2)} %)")
duplicados_por_id = df.duplicated(subset=['Patient ID'])
print(f"Pacientes con ID duplicado: {duplicados_por_id.sum()}")
valores_nulos = df.isnull().sum()
porcentaje_valores_nulos = (df.isnull().sum() / len(df) * 100).round(2)
nulos = pd.DataFrame({
    'Valores Nulos': valores_nulos,
    'Porcentaje (%)': porcentaje_valores_nulos
})
print("Categorias con valores nulos y su respectivo porcentaje: ")
print(nulos[nulos['Valores Nulos'] > 0])
print("\nValores presión sanguínea: ", df["Blood Pressure (mmHg)"].unique()[:10])
print("\nValores máximos y mínimos de las variables clínicas:")
print("Edad: ", df["Age"].min(), ",", df["Age"].max())
print("Peso: ", df["Weight (kg)"].min(), ",", df["Weight (kg)"].max())
print("Altura: ", df["Height (m)"].min(), ",", df["Height (m)"].max())
print("Presión sistólica: ",df["Systolic BP"].min(), ",", df["Systolic BP"].max())
print("Presión diastólica: ", df["Diastolic BP"].min(), ",", df["Diastolic BP"].max())
print("CVD Risk Score: ", df["CVD Risk Score"].min(), ",", df["CVD Risk Score"].max())
print("Estimated LDL (mg/dL): ", df["Estimated LDL (mg/dL)"].min(), ",", df["Estimated LDL (mg/dL)"].max())


Registros duplicados del data set: 151 (9.21 %)
Pacientes con ID duplicado: 263
Categorias con valores nulos y su respectivo porcentaje: 
                              Valores Nulos  Porcentaje (%)
Age                                      68            4.15
Weight (kg)                              73            4.45
Height (m)                               61            3.72
BMI                                      53            3.23
Abdominal Circumference (cm)             61            3.72
Total Cholesterol (mg/dL)                68            4.15
HDL (mg/dL)                              82            5.00
Fasting Blood Sugar (mg/dL)              54            3.29
Height (cm)                              68            4.15
Waist-to-Height Ratio                    76            4.64
Systolic BP                              61            3.72
Diastolic BP                             85            5.19
Estimated LDL (mg/dL)                    57            3.48
CVD Risk Score        

Sobre estas nuevas líneas de código determinamos lo siguiente:
1. Hay 151 registros completamente duplicados, lo que representa el 9.21% del dataset, dichos registros duplicados deben ser eliminados para evitar sesgos en el entrenamiento del modelo.
2. Existen 263 pacientes con IDs duplicados, esto puede traducirse en diferentes escenarios como, registros completamente duplicados, múltiples visitas del paciente (diferente fecha), o error de identificación.
2. Como lo mencionamos anteriormente, la variable **Blood Pressure** es un string que combina las variables **Systolic BP** y **Diastolic BP**, así que esta variable es redundante y podríamos eliminarla.
3. Se observan valores decimales en la variable **edad** (6.13 años), cosa que podría indicar errores de registro o transformaciones previas en los datos.
4. El valor mínimo de la variable peso es de 13.26kg, el de edad es 6.13 años, el de BMI es de 4.3, el de CVD Risk Score y Estimated LDL es negativo, estos valores son imposibles dado que estamos hablando de un estudio cardiovascular de adultos.
5. El valor máximos de la presión sistólica fue de 202.711, dicho valor podría ser un outlier.
6. Hay valores nulos en múltiples variables clínicas, cosa que requiere estrategias de imputación en la fase de preparación de datos, adicionalmente, la variable objetivo CVD Risk Score presenta 1.77% de valores nulos, dichos valores deberan ser tratados con cuidado.

## Preparación de datos 

Las principales irregularidades que encontramos en el set de datos fueron las siguientes:

1. 151 registros completamente duplicados.
2. 263 pacientes con IDs duplicados.
3. 29 valores nulos en nuestra variable objetivo **CVD Risk Score**.
4. Valores nulos en 14 columnas.
5. Categorias redundantes como **Blood Pressure (mmHg)**, **Height (cm)** y **CVD Risk Level**.
6. Valores imposibles en categorias como **Weight (kg)**, **Age**, **BMI**, **CVD Risk Score** y **Estimated LDL (mg/dL)**.
7. Tipos de datos erroneos respecto a la documentación, en especifico se espera que las categorias **Smoking Status**, **Diabetes Status** y **Family History of CVD** sean booleanos y no objects, de igual forma, **Date of Service** presenta multiples formatos, y se espera que sea tipo date y no object.

### Decisiones de limpieza de datos

Dadas estas irregularidades, vamos a tomar las siguientes decisiones teniendo en cuenta que vamos a construir un modelo de regresión lineal:

1. Los registros completamente duplicados los vamos a eliminar, dejando únicamente la primera ocurrencia del registro.

In [38]:
df_clean = df.copy()
before = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first')
after = len(df_clean)

print(f"Registros eliminados: {before - after}")
print(f"Dimensiones actuales: {df_clean.shape}")

Registros eliminados: 151
Dimensiones actuales: (151, 24)


2. Los pacientes con IDs duplicados los vamos a manejar de la siguiente manera, al estudiar los IDs duplicados nos dimos cuenta que todas las demás columnas son identicas en algunos casos, y en otros lo unico que cambia es el **CVD Risk Score**. Por tanto decidimos únicamente quedarnos con la primera ocurrencia del ID, pues hacer un promedio basado en 2 CVD Risk Scores diferentes a pesar de que todas las columnas son iguales podría sesgar al modelo.

In [39]:
ids_duplicados = df_clean[df_clean['Patient ID'].duplicated(keep=False)]
print("Exploración de IDs duplicados: ")
print("\n")
print(ids_duplicados.sort_values('Patient ID').head(10))
before = len(df_clean['Patient ID'])
df_clean = df_clean.drop_duplicates(subset=['Patient ID'], keep='first')
after = len(df_clean['Patient ID'])
print(f"\nRegistros eliminados: {before - after}")
print(f"Dimensiones actuales: {df_clean.shape}")

Exploración de IDs duplicados: 


Empty DataFrame
Columns: [Patient ID, Date of Service, Sex, Age, Weight (kg), Height (m), BMI, Abdominal Circumference (cm), Blood Pressure (mmHg), Total Cholesterol (mg/dL), HDL (mg/dL), Fasting Blood Sugar (mg/dL), Smoking Status, Diabetes Status, Physical Activity Level, Family History of CVD, Height (cm), Waist-to-Height Ratio, Systolic BP, Diastolic BP, Blood Pressure Category, Estimated LDL (mg/dL), CVD Risk Score, CVD Risk Level]
Index: []

[0 rows x 24 columns]

Registros eliminados: 0
Dimensiones actuales: (151, 24)


3. Los 29 valores nulos de la variable objetivo **CVD Risk Score** fueron removidos al eliminar los duplicados del set de datos.

In [34]:
valores_nulos = df_clean.isnull().sum()
porcentaje_valores_nulos = (df_clean.isnull().sum() / len(df) * 100).round(2)
nulos = pd.DataFrame({
    'Valores Nulos': valores_nulos,
    'Porcentaje (%)': porcentaje_valores_nulos
})
print("Categorias con valores nulos y su respectivo porcentaje: ")
print(nulos[nulos['Valores Nulos'] > 0])

Categorias con valores nulos y su respectivo porcentaje: 
                              Valores Nulos  Porcentaje (%)
Age                                      60           19.87
Weight (kg)                              66           21.85
Height (m)                               53           17.55
BMI                                      46           15.23
Abdominal Circumference (cm)             49           16.23
Total Cholesterol (mg/dL)                61           20.20
HDL (mg/dL)                              73           24.17
Fasting Blood Sugar (mg/dL)              48           15.89
Height (cm)                              60           19.87
Waist-to-Height Ratio                    67           22.19
Systolic BP                              55           18.21
Diastolic BP                             70           23.18
Estimated LDL (mg/dL)                    51           16.89
